# Clothing Classification Model Training (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_REPO/train_clothing_classifier_colab.ipynb)

This notebook trains a CNN classifier for garment type detection optimized for Google Colab.

**Features:**
- 🚀 GPU acceleration (automatically detected)
- 📦 Loads dataset from Google Drive
- 💾 Saves trained models back to Drive
- 📊 TensorFlow 2.15+ compatible models
- ⚡ Fast training with data augmentation

**Expected Folder Structure:**
```
My Drive/
  └── clothing_models/
      ├── dataset/
      │   └── clothing.zip (downloaded automatically)
      └── trained_models/
          ├── best_clothing_model.h5
          ├── clothing_model_final.h5
          ├── class_labels.json
          ├── model_config.json
          └── rejection_threshold.json
```

## 1. Setup Google Colab Environment

In [ ]:
# Check GPU availability
import tensorflow as tf

print("="*60)
print("Google Colab Environment Check")
print("="*60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Python version: {__import__('sys').version}")

# Check for GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"\n✅ GPU available: {len(gpu_devices)} device(s)")
    for i, gpu in enumerate(gpu_devices):
        print(f"   GPU {i}: {gpu}")
    print("\n🚀 Training will use GPU acceleration!")
else:
    print("\n⚠️ No GPU found. Training will use CPU (slower).")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")

print("="*60)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project directories in Drive
DRIVE_ROOT = '/content/drive/MyDrive/clothing_models'
DATASET_DIR = f'{DRIVE_ROOT}/dataset'
MODELS_DIR = f'{DRIVE_ROOT}/trained_models'

# Create directories if they don't exist
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"\n✅ Google Drive mounted")
print(f"📁 Project root: {DRIVE_ROOT}")
print(f"📁 Dataset directory: {DATASET_DIR}")
print(f"📁 Models directory: {MODELS_DIR}")

## 3. Download Dataset from Google Drive

**Dataset Link:** https://drive.google.com/file/d/1mqnxghcVu2trYKdbQRnd93OgvrEQJeZW/view

The dataset will be downloaded to your Google Drive and cached for future runs.

In [ ]:
# Install gdown for downloading from Google Drive
!pip install -q gdown

import gdown
import zipfile

# Dataset configuration
DATASET_URL = 'https://drive.google.com/uc?id=1mqnxghcVu2trYKdbQRnd93OgvrEQJeZW'
ZIP_PATH = f'{DATASET_DIR}/clothing.zip'
EXTRACT_PATH = f'{DATASET_DIR}/clothing_dataset'

# Download dataset if not already present
if not os.path.exists(ZIP_PATH):
    print("📥 Downloading dataset from Google Drive...")
    print(f"   This may take a few minutes...\n")
    gdown.download(DATASET_URL, ZIP_PATH, quiet=False)
    print(f"\n✅ Dataset downloaded: {ZIP_PATH}")
else:
    print(f"✅ Dataset already exists: {ZIP_PATH}")

# Extract dataset
if not os.path.exists(EXTRACT_PATH):
    print(f"\n📦 Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print(f"✅ Dataset extracted to: {EXTRACT_PATH}")
else:
    print(f"✅ Dataset already extracted: {EXTRACT_PATH}")

# Verify extraction
dataset_root = EXTRACT_PATH
subdirs = [d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))]

print(f"\n📊 Dataset structure:")
print(f"   Root: {dataset_root}")
print(f"   Classes found: {subdirs}")

## 4. Install Required Packages

In [ ]:
# Install any additional packages needed
!pip install -q scikit-learn matplotlib

# Import libraries
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All packages imported successfully")

## 5. Discover and Verify Classes

In [ ]:
# Discover classes from dataset
all_dirs = [d for d in os.listdir(dataset_root)
            if os.path.isdir(os.path.join(dataset_root, d))]

# Verify required classes
required = {'tshirt', 'trousers'}
missing = required - set(all_dirs)
if missing:
    raise ValueError(f"❌ Missing required class folders: {missing}")

# Build class list (stable order for consistency)
classes = ['trousers', 'tshirt']  # Keep stable order
if 'other' in all_dirs:
    classes.append('other')

num_classes = len(classes)

print("="*60)
print("Dataset Classes")
print("="*60)
print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")
print("\n📊 Images per class:")

# Count images per class
total_images = 0
for cls in classes:
    cls_path = os.path.join(dataset_root, cls)
    n_images = len([f for f in os.listdir(cls_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
    total_images += n_images
    print(f"  {cls:12s}: {n_images:5d} images")

print(f"\n📊 Total images: {total_images}")
print("="*60)

## 6. Create Data Generators

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Larger batch size for Colab GPU

# Training generator with aggressive augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation generator (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Create generators
train_gen = train_datagen.flow_from_directory(
    dataset_root,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
    classes=classes
)

val_gen = val_datagen.flow_from_directory(
    dataset_root,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
    classes=classes
)

print("="*60)
print("Data Generators")
print("="*60)
print(f"Training samples:   {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Batch size:         {BATCH_SIZE}")
print(f"Image size:         {IMG_SIZE}")
print(f"\nClass indices: {train_gen.class_indices}")
print("="*60)

## 7. Compute Class Weights

In [ ]:
# Count images per class
class_counts = {}
for cname, cidx in train_gen.class_indices.items():
    cpath = os.path.join(dataset_root, cname)
    n = len([f for f in os.listdir(cpath)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
    class_counts[cidx] = n

# Compute balanced weights
classes_np = np.array(list(class_counts.keys()))
counts_np = np.array(list(class_counts.values()))
cw = compute_class_weight(
    'balanced',
    classes=classes_np,
    y=np.repeat(classes_np, counts_np)
)
class_weight = dict(zip(classes_np, cw))

print("="*60)
print("Class Weights (Balancing)")
print("="*60)
for idx, weight in class_weight.items():
    cls_name = [k for k, v in train_gen.class_indices.items() if v == idx][0]
    count = class_counts[idx]
    print(f"{cls_name:12s} (index {idx}): weight={weight:.4f}, count={count}")
print("="*60)

## 8. Build CNN Model

In [ ]:
def create_model(input_shape, num_classes):
    """Create CNN model for garment classification."""
    model = Sequential([
        # Block 1: 32 filters
        Conv2D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.2),

        # Block 2: 64 filters
        Conv2D(64, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.3),

        # Block 3: 128 filters
        Conv2D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.3),

        # Block 4: 256 filters
        Conv2D(256, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2),
        Dropout(0.4),

        # Dense layers
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),

        # Output layer
        Dense(num_classes, activation=('softmax' if num_classes >= 3 else 'sigmoid'))
    ])
    return model

# Create model
model = create_model(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    num_classes=num_classes
)

# Determine head type and loss function
if num_classes >= 3:
    loss = 'categorical_crossentropy'
    head_type = 'softmax'
else:
    loss = 'binary_crossentropy'
    head_type = 'sigmoid_ovr'

# Compile model
model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss=loss,
    metrics=['accuracy']
)

print("="*60)
print("Model Architecture")
print("="*60)
print(f"Head type: {head_type}")
print(f"Loss function: {loss}")
print(f"Optimizer: Adam (lr=5e-4)")
print("="*60)
model.summary()

## 9. Save Configuration Files

In [ ]:
# Save class labels
labels_path = f'{MODELS_DIR}/class_labels.json'
with open(labels_path, 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print(f"✅ Saved: {labels_path}")
print(json.dumps(train_gen.class_indices, indent=2))

# Save model config
config_path = f'{MODELS_DIR}/model_config.json'
with open(config_path, 'w') as f:
    json.dump({'head_type': head_type, 'img_size': IMG_SIZE[0]}, f, indent=2)
print(f"\n✅ Saved: {config_path}")
print(f"   head_type: {head_type}")
print(f"   img_size: {IMG_SIZE[0]}")

## 10. Setup Callbacks

In [ ]:
# Callback paths in Google Drive (will be saved automatically)
best_model_path = f'{MODELS_DIR}/best_clothing_model.h5'

callbacks = [
    ModelCheckpoint(
        best_model_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-5,
        verbose=1
    )
]

print("✅ Callbacks configured:")
print(f"   - Best model will be saved to: {best_model_path}")
print(f"   - Early stopping patience: 15 epochs")
print(f"   - Learning rate reduction on plateau")

## 11. Train Model 🚀

**Training will start now!**

With GPU: ~10-20 minutes
Without GPU: ~30-60 minutes

Go grab a coffee! ☕

In [ ]:
EPOCHS = 50
steps_per_epoch = max(1, train_gen.samples // BATCH_SIZE)
val_steps = max(1, val_gen.samples // BATCH_SIZE)

print("="*60)
print("Starting Training")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Validation steps: {val_steps}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Using: {'GPU 🚀' if tf.config.list_physical_devices('GPU') else 'CPU'}")
print("="*60)
print("\n🚀 Training started...\n")

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=val_steps,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

print("\n" + "="*60)
print("✅ Training completed!")
print("="*60)

## 12. Save Final Model

In [ ]:
# Save final model
final_model_path = f'{MODELS_DIR}/clothing_model_final.h5'
model.save(final_model_path)
print(f"✅ Saved final model: {final_model_path}")

# Verify all model files
print("\n📁 Saved model files in Google Drive:")
model_files = [
    'best_clothing_model.h5',
    'clothing_model_final.h5',
    'class_labels.json',
    'model_config.json'
]

for fname in model_files:
    fpath = f'{MODELS_DIR}/{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✅ {fname:30s} ({size_mb:6.2f} MB)")
    else:
        print(f"  ❌ {fname:30s} (missing)")

## 13. Estimate Rejection Threshold

In [ ]:
try:
    # Get validation batch for threshold estimation
    val_batch = next(iter(val_gen))
    probs = model.predict(val_batch[0], verbose=0)
    maxp = probs.max(axis=1)
    
    # Conservative threshold at 10th percentile
    tau = max(0.6, float(np.quantile(maxp, 0.10)))
    
    # Save threshold
    tau_path = f'{MODELS_DIR}/rejection_threshold.json'
    with open(tau_path, 'w') as f:
        json.dump({'tau': tau}, f, indent=2)
    
    print("="*60)
    print("Rejection Threshold Estimation")
    print("="*60)
    print(f"Estimated tau: {tau:.4f}")
    print(f"Saved to: {tau_path}")
    print("\n📊 Validation confidence distribution:")
    print(f"   Min:    {maxp.min():.4f}")
    print(f"   10%:    {np.quantile(maxp, 0.10):.4f}  ← threshold")
    print(f"   Median: {np.quantile(maxp, 0.50):.4f}")
    print(f"   90%:    {np.quantile(maxp, 0.90):.4f}")
    print(f"   Max:    {maxp.max():.4f}")
    print("="*60)
    
except Exception as e:
    print(f"⚠️ Could not estimate tau: {e}")
    tau = 0.75
    with open(f'{MODELS_DIR}/rejection_threshold.json', 'w') as f:
        json.dump({'tau': tau}, f, indent=2)
    print(f"Using default tau = {tau}")

## 14. Visualize Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot accuracy
ax1.plot(history.history['accuracy'], label='Training', linewidth=2.5, color='#2E86AB')
ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2.5, color='#A23B72')
ax1.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax1.set_title('Model Accuracy', fontsize=15, fontweight='bold', pad=15)
ax1.legend(fontsize=11, loc='lower right')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_ylim([0, 1])

# Plot loss
ax2.plot(history.history['loss'], label='Training', linewidth=2.5, color='#2E86AB')
ax2.plot(history.history['val_loss'], label='Validation', linewidth=2.5, color='#A23B72')
ax2.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax2.set_ylabel('Loss', fontsize=13, fontweight='bold')
ax2.set_title('Model Loss', fontsize=15, fontweight='bold', pad=15)
ax2.legend(fontsize=11, loc='upper right')
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plot_path = f'{MODELS_DIR}/training_history.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Saved training plot: {plot_path}")

# Print final metrics
print("\n" + "="*60)
print("Final Training Metrics")
print("="*60)
print(f"Training Accuracy:   {history.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Training Loss:       {history.history['loss'][-1]:.4f}")
print(f"Validation Loss:     {history.history['val_loss'][-1]:.4f}")
print("="*60)

## 15. Test Model Loading (TF 2.15+ Compatibility)

In [ ]:
print("="*60)
print("Testing Model Loading")
print("="*60)

# Test loading best model
try:
    test_model = tf.keras.models.load_model(best_model_path, compile=False)
    print(f"✅ {os.path.basename(best_model_path)} loads successfully!")
    print(f"   Input shape:  {test_model.input_shape}")
    print(f"   Output shape: {test_model.output_shape}")
    
    # Test prediction
    dummy_input = np.random.rand(1, 224, 224, 3).astype('float32')
    prediction = test_model.predict(dummy_input, verbose=0)
    print(f"\n✅ Test prediction successful!")
    print(f"   Probabilities: {prediction[0]}")
    print(f"   Predicted class: {np.argmax(prediction[0])} ({classes[np.argmax(prediction[0])]})")
    print(f"   Max confidence: {prediction[0].max():.4f}")
    print("\n🎉 Model is compatible with TensorFlow 2.15+!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nIf you see a 'batch_shape' error, you need to retrain with TF 2.13+")

print("="*60)

## 16. Download Models (Optional)

If you want to download the trained models to your computer:

In [ ]:
"""
Improved download logic for Google Colab - Works even if training was interrupted!
"""
from google.colab import files
import zipfile

print("="*60)
print("📦 Preparing Models for Download")
print("="*60)

# Check what files exist
available_files = []
all_files = [
    ('best_clothing_model.h5', 'Best model (highest validation accuracy)'),
    ('clothing_model_final.h5', 'Final model (last epoch)'),
    ('class_labels.json', 'Class name mappings'),
    ('model_config.json', 'Model configuration'),
    ('rejection_threshold.json', 'Confidence threshold'),
    ('training_history.png', 'Training plots')
]

print("\n📊 Checking available files in Google Drive:")
for fname, description in all_files:
    fpath = f'{MODELS_DIR}/{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✅ {fname:30s} ({size_mb:6.2f} MB) - {description}")
        available_files.append(fname)
    else:
        print(f"  ❌ {fname:30s} (not found) - {description}")

if not available_files:
    print("\n❌ No model files found!")
    print("   Make sure training has started and saved at least one model.")
    print(f"   Looking in: {MODELS_DIR}")
else:
    print(f"\n✅ Found {len(available_files)} file(s) ready for download")

    # Ask user what to download
    print("\n" + "="*60)
    print("Download Options:")
    print("="*60)
    print("1. Download ALL available files (recommended)")
    print("2. Download only essential files (models + configs)")
    print("3. Download only best_clothing_model.h5 (smallest)")
    print("="*60)

    choice = input("\nEnter your choice (1-3) [default: 1]: ").strip() or "1"

    # Determine which files to include
    if choice == "1":
        files_to_download = available_files
        print("\n📦 Downloading ALL files...")
    elif choice == "2":
        files_to_download = [f for f in available_files if f.endswith(('.h5', '.json'))]
        print("\n📦 Downloading essential files (models + configs)...")
    elif choice == "3":
        files_to_download = ['best_clothing_model.h5'] if 'best_clothing_model.h5' in available_files else []
        print("\n📦 Downloading best model only...")
    else:
        print("❌ Invalid choice, defaulting to ALL files")
        files_to_download = available_files

    if not files_to_download:
        print("❌ No files selected for download")
    else:
        # Create zip file
        zip_filename = '/content/trained_models.zip'

        print(f"\n📦 Creating zip archive...")
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for fname in files_to_download:
                fpath = f'{MODELS_DIR}/{fname}'
                zipf.write(fpath, fname)
                print(f"  ✅ Added: {fname}")

        zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
        print(f"\n✅ Zip file created successfully!")
        print(f"   Location: {zip_filename}")
        print(f"   Size: {zip_size_mb:.2f} MB")
        print(f"   Contains: {len(files_to_download)} file(s)")

        # Download
        print("\n💾 Starting download to your computer...")
        print("   (Check your browser's download folder)")
        try:
            files.download(zip_filename)
            print("\n🎉 Download completed successfully!")
        except Exception as e:
            print(f"\n⚠️ Download failed: {e}")
            print("\nAlternative: The zip file is saved at:")
            print(f"   {zip_filename}")
            print("\nYou can download it manually from the Files panel (left sidebar)")

        # Instructions
        print("\n" + "="*60)
        print("📝 Next Steps:")
        print("="*60)
        print("1. Extract the downloaded 'trained_models.zip'")
        print("2. Copy files to your FastAPI backend:")
        print("   cp *.h5 *.json /path/to/image-extraction-backend/models/")
        print("\n3. Test the model:")
        print("   cd /path/to/image-extraction-backend")
        print("   python test_model_load.py")
        print("\n4. If test passes, commit and deploy to Railway!")
        print("="*60)

        # Show what was downloaded
        print("\n📋 Files in trained_models.zip:")
        for fname in files_to_download:
            print(f"  - {fname}")

        print("\n✨ Your models are ready for deployment! 🚀")

## 🎉 Training Complete!

### ✅ What was created:
1. **best_clothing_model.h5** - Best model (saved in Google Drive)
2. **clothing_model_final.h5** - Final model after all epochs
3. **class_labels.json** - Class mappings
4. **model_config.json** - Model configuration
5. **rejection_threshold.json** - Confidence threshold
6. **training_history.png** - Training visualization

### 📂 All files are saved in:
`/content/drive/MyDrive/clothing_models/trained_models/`

### 🚀 Next Steps:
1. Download the `trained_models.zip` file (already done above)
2. Extract and copy files to your FastAPI backend:
   ```bash
   # In your project directory
   unzip trained_models.zip
   cp *.h5 *.json image-extraction-backend/models/
   ```
3. Test the model:
   ```bash
   python test_model_load.py
   ```
4. Deploy to Railway with the new models!

### ✨ Key Features:
- ✅ TensorFlow 2.15+ compatible (no `batch_shape` errors)
- ✅ Class-weighted training (handles imbalance)
- ✅ Data augmentation (improves generalization)
- ✅ Early stopping (prevents overfitting)
- ✅ Automatic threshold estimation

**Happy deploying! 🚀**

## 21. Final Summary & Export Results

In [ ]:
# FIGURE 3: Precision-Recall Curves
print("📈 Figure 3: Precision-Recall Curves...")
fig, ax = plt.subplots(figsize=(10, 8))
for i, (cls, color) in enumerate(zip(classes, colors)):
    precision, recall, _ = precision_recall_curve(y_true_onehot[:, i], y_prob[:, i])
    ap = average_precision_score(y_true_onehot[:, i], y_prob[:, i])
    ax.plot(recall, precision, color=color, lw=2.5,
            label=f'{cls.capitalize()} (AP = {ap:.3f})')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall', fontweight='bold', fontsize=12)
ax.set_ylabel('Precision', fontweight='bold', fontsize=12)
ax.set_title('Precision-Recall Curves', fontweight='bold', fontsize=14, pad=15)
ax.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig3_precision_recall.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f'{OUTPUT_DIR}/fig3_precision_recall.pdf', bbox_inches='tight')
plt.show()

# FIGURE 4: Per-Class Metrics Bar Chart
print("\n📈 Figure 4: Per-Class Metrics...")
class_report = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(classes))
width = 0.25
precision_vals = [class_report[cls]['precision'] for cls in classes]
recall_vals = [class_report[cls]['recall'] for cls in classes]
f1_vals = [class_report[cls]['f1-score'] for cls in classes]
bars1 = ax.bar(x - width, precision_vals, width, label='Precision', color='#3498db', edgecolor='black', linewidth=0.7)
bars2 = ax.bar(x, recall_vals, width, label='Recall', color='#2ecc71', edgecolor='black', linewidth=0.7)
bars3 = ax.bar(x + width, f1_vals, width, label='F1-Score', color='#e74c3c', edgecolor='black', linewidth=0.7)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xlabel('Class', fontweight='bold', fontsize=12)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Per-Class Performance Metrics', fontweight='bold', fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels([cls.capitalize() for cls in classes], fontsize=11)
ax.legend(fontsize=11, frameon=True, shadow=True, loc='lower right')
ax.set_ylim([0, 1.1])
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig4_per_class_metrics.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f'{OUTPUT_DIR}/fig4_per_class_metrics.pdf', bbox_inches='tight')
plt.show()

print("✅ Figures 3-4 complete!")

In [ ]:
print("📈 Generating Figure 1: Confusion Matrices...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[c.capitalize() for c in classes],
            yticklabels=[c.capitalize() for c in classes],
            ax=axes[0], cbar_kws={'label': 'Count'},
            linewidths=0.5, linecolor='gray')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold', pad=15)
axes[0].set_xlabel('Predicted Label', fontweight='bold')
axes[0].set_ylabel('True Label', fontweight='bold')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=[c.capitalize() for c in classes],
            yticklabels=[c.capitalize() for c in classes],
            ax=axes[1], cbar_kws={'label': 'Percentage'},
            linewidths=0.5, linecolor='gray', vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold', pad=15)
axes[1].set_xlabel('Predicted Label', fontweight='bold')
axes[1].set_ylabel('True Label', fontweight='bold')

plt.tight_layout()
fig1_path = f'{OUTPUT_DIR}/fig1_confusion_matrices.png'
plt.savefig(fig1_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig1_path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f"✅ Saved: {fig1_path}")

# ============================================================================
print("\n📈 Generating Figure 2: ROC Curves...")

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set2(np.linspace(0, 1, num_classes))

for i, (cls, color) in enumerate(zip(classes, colors)):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
    roc_auc_cls = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f'{cls.capitalize()} (AUC = {roc_auc_cls:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
ax.set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
ax.set_title('ROC Curves (One-vs-Rest)', fontweight='bold', fontsize=14, pad=15)
ax.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
fig2_path = f'{OUTPUT_DIR}/fig2_roc_curves.png'
plt.savefig(fig2_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig2_path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f"✅ Saved: {fig2_path}")

In [ ]:
print("🔮 Generating predictions on validation set...")

# Generate predictions
y_prob = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = val_gen.classes

# Get one-hot encoded labels
y_true_onehot = np.eye(num_classes)[y_true]

# Calculate basic masks
correct_mask = y_pred == y_true
max_probs = y_prob.max(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-12)

print(f"\n✅ Predictions complete")
print(f"   Total samples: {len(y_true)}")
print(f"   Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"   Classes: {classes}")

In [ ]:
# Install additional packages for evaluation
!pip install -q seaborn scipy

import seaborn as sns
import itertools
import time
from scipy import stats
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    log_loss, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    brier_score_loss, cohen_kappa_score
)
from sklearn.calibration import calibration_curve
from tensorflow.keras.models import load_model

# Set publication-quality plot style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
})

# Create output directory for thesis figures
OUTPUT_DIR = f'{MODELS_DIR}/thesis_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*70)
print("📊 THESIS EVALUATION SETUP")
print("="*70)
print(f"Output directory: {OUTPUT_DIR}")
print(f"All figures will be saved at 300 DPI")
print("="*70)

print("\n✅ All sklearn metrics imported successfully:")
print("   - accuracy_score")
print("   - classification_report")
print("   - confusion_matrix")
print("   - log_loss")
print("   - roc_auc_score, roc_curve, auc")
print("   - precision_recall_curve, average_precision_score")
print("   - brier_score_loss")
print("   - cohen_kappa_score")
print("\n📋 Ready for evaluation cells below!")

---

# 📊 THESIS EVALUATION - Publication Quality Analysis

**The following cells generate comprehensive evaluation figures for your thesis:**

- Confusion matrices & classification metrics
- ROC & Precision-Recall curves
- Calibration plots & reliability analysis
- Error analysis & confidence distributions
- Model efficiency metrics
- Threshold sensitivity analysis
- Dataset distribution analysis
- Statistical significance tests

**All figures are saved at 300 DPI in both PNG and PDF formats.**

Run all cells below to generate complete thesis materials!